# 12d. Joint Optimization with KDE Likelihood + Fisher Information

**Goal:** replace 12c's W1/mean-matching losses with a proper **negative log-likelihood** — a 2D KDE over (FWHM, σ) — turning the optimization into an MLE. That makes the **Fisher information / Cramér–Rao bound** applicable as the uncertainty statement.

| Gradient | Source | Why |
|---|---|---|
| **μ** (weighted REINFORCE) | ∇μL = −Σ_i (B_i − B̄)·(n_i−μ)/σ² | kernel responsibilities give a smooth, self-normalizing advantage (no EMA baseline, no mean-matching term) |
| **γ** (analytic KDE chain) | ∇γL = −(1/N)Σ_k Σ_i w̄_k,i·[(f_k−f_i)·df_i/dγ/h_f² + (s_k−s_i)·ds_i/dγ/h_s²] | chain rule through the kernel; df/dγ via implicit diff, ds/dγ via CRLB |
| **Fisher** | J(θ̂) = (1/N)Σ_k s_k s_kᵀ | empirical Fisher → CRB std for μ̂ and γ̂ |

Everything reuses `src/` unchanged: `draw_fixed_noise`, `compute_fwhm_and_dgamma`, `fit_profile`.

In [2]:
!ls

01-mc-algorithm.ipynb
02-sampling-toy.ipynb
03-fitting-toy-fwmh.ipynb
03-fitting-toy.ipynb
04-eda.ipynb
05-loss-mmd.ipynb
05-loss-w1.ipynb
06-gamma.ipynb
06-gamma-simple.ipynb
07-reinforce-toy.ipynb
08-reinforce-per-run.ipynb
09-mu-sigma-fwhm-map.ipynb
10-joint-optimization.ipynb
11-fwhm-pairplot.ipynb
12a-joint-opt-with-sigma-noise.ipynb
12b-joint-opt-with-sigma-lr-decay.ipynb
12c-joint-opt-with-sigma-mean-fwhm.ipynb
12d-joint-opt-likelihood-fisher.ipynb
13a-diagnose.ipynb
13b-fix-quantile-bug.ipynb
13b-quantile-fix-executed.ipynb
13b-quantile-fix.ipynb
13c-faster-gamma.ipynb
13c-more-iterations.ipynb
13c-reduce-sigma-weight-executed.ipynb
13c-reduce-sigma-weight.ipynb
13d-higher-gamma-lr-executed.ipynb
13d-higher-gamma-lr.ipynb
13e-moderate-gamma-lr-executed.ipynb
13e-moderate-gamma-lr.ipynb
13f-higher-background-noise-executed.ipynb
13f-higher-background-noise.ipynb
13-real-data-optimization-executed.ipynb
13-real-data-optimization-output.ipynb


In [ ]:

import math, time
import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float32)

from src.fitting import (
    _raw_from_width as _rw,
    log_pdf, nll, fwhm_from_theta, fit_profile
)
from src.samplers import draw_fixed_noise, build_photons
from src.implicit import compute_fwhm_and_dgamma

print('Imports OK')

ImportError: attempted relative import with no known parent package

In [ ]:
GAMMA_TRUE = 20.0
NBAR_TRUE = 50.0
LAMBDA_ = 2.0

N_TARGET = 200
N_RUNS = 500            # simulated runs per iteration (M)
N_ITER = 80

SIGMA_PROP = 6.0          # physical noise std
LR_MU = 15.0              # max learning rate for mu (decays to floor)
LR_GAMMA = 0.5            # learning rate for gamma
CLIP = 10.0               # gradient clipping

MU_INIT = 8.0
GAMMA_INIT = 5.0
SEED = 42

# --- 12d additions ---
M_FINAL = 5000            # runs for the final Fisher block
FISHER_SEEDS = 5          # independent seeds averaged for the Fisher estimate
FISHER_BASE_SEED = 7000

print('Parameters set')

## Generate Target Data

Collect both FWHM and σ_FWHM at true (μ=50, γ=20) — the 2D observed distribution.
The KDE bandwidths h_f, h_s use **Scott's rule**, computed once on the target data and kept fixed for the whole run (otherwise the likelihood would change meaning).

In [ ]:
t_total = time.time()

def _fit_fn(ph):
    return fit_profile(ph, n_iters=80, model='lorentzian', uniform_bg=False)

def _fwhm_fn(th):
    return fwhm_from_theta(th, model='lorentzian')

def _nll_fn(th, ph):
    return nll(th, ph, model='lorentzian', uniform_bg=False)

print(f"Generating target ({N_TARGET} runs)...", end=" ", flush=True)
rng = np.random.default_rng(SEED)
target_fwhms, target_sigmas = [], []

for ti in range(N_TARGET):
    if ti % 100 == 0:
        print(f'{ti}...', end=' ', flush=True)
    u, b, n = draw_fixed_noise(NBAR_TRUE, SIGMA_PROP, LAMBDA_, rng)
    fw, sig, _ = compute_fwhm_and_dgamma(
        GAMMA_TRUE, u.numpy(), b.numpy(),
        _fit_fn, _fwhm_fn, _nll_fn, n_params=2
    )
    target_fwhms.append(fw)
    target_sigmas.append(sig)

target_f = torch.tensor(target_fwhms, dtype=torch.float32)
target_s = torch.tensor(target_sigmas, dtype=torch.float32)

# Scott's rule bandwidths: h_j = sigma_hat_j * N^(-1/(d+4)), d = 2.
SCOTT_EXP = N_TARGET ** (-1.0 / 6.0)
H_F = float(target_f.std()) * SCOTT_EXP
H_S = float(target_s.std()) * SCOTT_EXP
print(f"\ntarget FWHM: {target_f.mean():.1f} ± {target_f.std():.1f}")
print(f"target σ:    {target_s.mean():.2f} ± {target_s.std():.2f}")
print(f"Scott bandwidths: h_f={H_F:.2f} MHz, h_s={H_S:.3f} MHz")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(target_f.numpy(), bins=40, density=True, alpha=0.7, color='#2d6a4f')
ax1.set_xlabel('FWHM (MHz)'); ax1.set_ylabel('Density')
ax1.set_title(f'Target FWHM (μ={NBAR_TRUE}, γ={GAMMA_TRUE})')
ax1.grid(alpha=0.2)
ax2.hist(target_s.numpy(), bins=40, density=True, alpha=0.7, color='#2d6a4f')
ax2.set_xlabel('σ_FWHM (MHz)'); ax2.set_ylabel('Density')
ax2.set_title('Target Fit Uncertainty')
ax2.grid(alpha=0.2)
plt.tight_layout()
plt.show()
print(f"({time.time()-t_total:.0f}s)")

## KDE Likelihood Loss & Gradients

Simulated runs `(f_i, s_i)`, i=1..M; observed data points `(f_k, s_k)`, k=1..N.

```
p̂(f, s; θ) = (1/M) Σ_i exp(−(f−f_i)²/2h_f² − (s−s_i)²/2h_s²)
L(θ) = −(1/N) Σ_k log p̂(f_k, s_k; θ)
```

Row-normalized kernel weights (the responsibilities — each kernel's share of the density at each data point):

```
w̄_k,i = K_k,i / Σ_j K_k,j
```

Per-data-point score vectors:
- **μ** (weighted REINFORCE): `s_μ,k = Σ_i w̄_k,i · (n_i − μ)/σ_phys²`
- **γ** (analytic KDE chain): `s_γ,k = Σ_i w̄_k,i · [ (f_k−f_i)·(df_i/dγ)/h_f² + (s_k−s_i)·(ds_i/dγ)/h_s² ]`

Gradient descent on L:
- `∇_μ L = −Σ_i (B_i − B̄)·(n_i−μ)/σ²`, with `B_i = (1/N)Σ_k w̄_k,i` (run responsibility, baseline = sample mean)
- `∇_γ L = −(1/N)Σ_k s_γ,k`

The μ gradient is REINFORCE because the sample position depends on the **discrete** photon count `n_i ~ round(N(μ, σ²))` — no chain rule exists there. The γ gradient is an ordinary chain rule through the kernel (df/dγ implicit, ds/dγ ≈ 2/√n CRLB).

In [ ]:
def kde_scores(sim_f, sim_s, sim_n, sim_df, sim_ds, data_f, data_s, h_f, h_s, mu):
    """
    2D KDE negative log-likelihood and per-data-point score vectors.

    sim_*  : (M,) per-run values:
        sim_f   fitted FWHM
        sim_s   fit uncertainty σ
        sim_n   photon count (for the REINFORCE score)
        sim_df  dFWHM/dgamma (implicit diff)
        sim_ds  dsigma/dgamma (CRLB approx)
    data_f, data_s : (N,) observed (FWHM, σ) points
    h_f, h_s : fixed KDE bandwidths

    Returns:
        s_mu    (N,)  per-data-point μ-score
        s_gamma (N,)  per-data-point γ-score
        nll     scalar mean negative log-likelihood
        w       (N, M) row-normalized kernel weights
    """
    d_f = data_f[:, None] - sim_f[None, :]                       # (N, M)
    d_s = data_s[:, None] - sim_s[None, :]                       # (N, M)
    W = torch.exp(-0.5 * (d_f / h_f) ** 2 - 0.5 * (d_s / h_s) ** 2)   # (N, M)
    w = W / W.sum(dim=1, keepdim=True).clamp_min(1e-12)          # row-normalized

    # μ: weighted REINFORCE score
    score = (sim_n[None, :] - mu) / SIGMA_PROP ** 2              # (1, M)
    s_mu = (w * score).sum(dim=1)                                # (N,)

    # γ: analytic chain through the kernel
    dlogG = (d_f * sim_df[None, :]) / h_f ** 2 + (d_s * sim_ds[None, :]) / h_s ** 2
    s_gamma = (w * dlogG).sum(dim=1)                             # (N,)

    # NLL value (kernel normalization constants cancel in gradients)
    logp = torch.log((W.sum(dim=1) / len(sim_f)).clamp_min(1e-30))
    nll_val = -logp.mean()
    return s_mu, s_gamma, nll_val, w


def kde_scores_1d(sim_f, sim_n, sim_df, data_f, h_f, mu):
    """
    1D FWHM-only version (degeneracy diagnostic): expected to produce a
    near-singular Fisher matrix, showing why the σ dimension is needed.
    """
    d_f = data_f[:, None] - sim_f[None, :]                       # (N, M)
    W = torch.exp(-0.5 * (d_f / h_f) ** 2)                       # (N, M)
    w = W / W.sum(dim=1, keepdim=True).clamp_min(1e-12)
    score = (sim_n[None, :] - mu) / SIGMA_PROP ** 2
    s_mu = (w * score).sum(dim=1)
    dlogG = (d_f * sim_df[None, :]) / h_f ** 2
    s_gamma = (w * dlogG).sum(dim=1)
    return s_mu, s_gamma, w

print('KDE likelihood functions ready')

## Joint Optimization Loop

Same outer loop as 12c: N_ITER=80 iterations × N_RUNS simulated runs, μ LR 15 → 4.5, γ LR 0.5, gradient clipping ±10, parameter clamps. Only the loss and gradients change.

In [ ]:
mu_val = float(MU_INIT)
gamma_val = float(GAMMA_INIT)
history = []

print(f"μ_init={MU_INIT}, γ_init={GAMMA_INIT}, true=({NBAR_TRUE},{GAMMA_TRUE})")
print(f"N_ITER={N_ITER}, N_RUNS={N_RUNS} (KDE likelihood)")
print(f"h_f={H_F:.2f}, h_s={H_S:.3f} (Scott, fixed)")
print(f"MU LR: linear decay {LR_MU} -> {0.3*LR_MU:.1f} over {N_ITER} steps\n")

for step in range(N_ITER):
    rng2 = np.random.default_rng(SEED + step)
    fwhms, sigmas, dfs, dsigmas, ns = [], [], [], [], []

    for _ in range(N_RUNS):
        u, b, n = draw_fixed_noise(mu_val, SIGMA_PROP, LAMBDA_, rng2)
        fw, sig, dg = compute_fwhm_and_dgamma(
            gamma_val, u.numpy(), b.numpy(),
            _fit_fn, _fwhm_fn, _nll_fn, n_params=2
        )
        fwhms.append(fw); sigmas.append(sig); dfs.append(dg)
        dsigmas.append(min(2.0 / math.sqrt(max(int(n), 1)), 2.0))   # CRLB dσ/dγ
        ns.append(n)

    ft = torch.tensor(fwhms, dtype=torch.float32)
    si_t = torch.tensor(sigmas, dtype=torch.float32)
    nt = torch.tensor(ns, dtype=torch.float32)
    dg_t = torch.tensor(dfs, dtype=torch.float32)
    ds_t = torch.tensor(dsigmas, dtype=torch.float32)

    # Per-data-point scores + NLL
    s_mu, s_gamma, nll_val, w = kde_scores(
        ft, si_t, nt, dg_t, ds_t,
        target_f, target_s, H_F, H_S, mu_val
    )

    # ---- μ gradient: weighted REINFORCE with responsibility baseline ----
    B = w.mean(dim=0)                                    # (M,) run responsibilities
    score = (nt - mu_val) / SIGMA_PROP ** 2              # (M,)
    grad_mu = float(max(min(-(B - B.mean()) @ score, CLIP), -CLIP))

    # ---- γ gradient: analytic KDE chain ----
    grad_gamma = float(max(min(-s_gamma.mean(), CLIP), -CLIP))

    # ---- updates: gradient descent on L ----
    lr_mu_decay = LR_MU * max(0.3, 1.0 - step / N_ITER)  # 15 -> 4.5
    mu_val -= lr_mu_decay * grad_mu
    mu_val = max(1.0, min(200.0, mu_val))
    gamma_val -= LR_GAMMA * grad_gamma
    gamma_val = max(0.1, min(100.0, gamma_val))

    info = {
        'step': step,
        'mu': mu_val, 'gamma': gamma_val,
        'nll': float(nll_val),
        'grad_mu': grad_mu, 'grad_gamma': grad_gamma,
        'mean_n': float(nt.mean()),
        'B_max': float(B.max()),
    }
    history.append(info)

    if step % 5 == 0 or step == N_ITER - 1:
        print(f"  S{step:2d}: μ={mu_val:6.2f} γ={gamma_val:5.1f} | "
              f"NLL={nll_val:7.2f} | ∇μ={grad_mu:+.4f} ∇γ={grad_gamma:+.4f} | "
              f"n̄={info['mean_n']:4.1f} Bmax={info['B_max']:.2f} "
              f"({time.time()-t_total:.0f}s)", flush=True)

print(f"\nDone. {time.time()-t_total:.0f}s")

## Results

In [ ]:
if len(history) > 0:
    fmu = history[-1]['mu']
    fga = history[-1]['gamma']
    print(f"{'='*70}")
    print(f"  JOINT OPTIMIZATION — 2D KDE LIKELIHOOD (FWHM + σ)")
    print(f"{'='*70}")
    print(f"  μ:     {MU_INIT:.0f} → {fmu:.2f}  (true={NBAR_TRUE})  error={abs(fmu-NBAR_TRUE):.2f}")
    print(f"  γ:     {GAMMA_INIT:.0f} → {fga:.2f}  (true={GAMMA_TRUE})  error={abs(fga-GAMMA_TRUE):.2f}")
    print(f"  NLL:   {history[0]['nll']:.2f} → {history[-1]['nll']:.2f}")
    print(f"  Time:  {time.time()-t_total:.0f}s")
    print(f"  12c reference: μ → 48.45, γ → 19.62 (W1 loss)")
    print(f"{'='*70}")

## Fisher Information & Cramér–Rao Bound

At the recovered (μ̂, γ̂), run a fresh high-M simulation and evaluate the per-data-point scores. The observed/empirical Fisher matrix is the average outer product:

```
J(θ̂) = (1/N) Σ_k s_k s_kᵀ ,   s_k = (s_μ,k, s_γ,k)
```

Then `Cov(θ̂) ≈ J(θ̂)⁻¹` → CRB standard errors, the μ–γ correlation, and a confidence ellipse. The 1D FWHM-only version is a degeneracy diagnostic: its Fisher should have a near-zero eigenvalue (the flat direction that motivated σ-matching).

In [ ]:
mu_final = history[-1]['mu']
gamma_final = history[-1]['gamma']

J_list, J1_list = [], []

for seed in range(FISHER_SEEDS):
    rng = np.random.default_rng(FISHER_BASE_SEED + seed)
    fwhms, sigmas, dfs, dsigmas, ns = [], [], [], [], []
    for _ in range(M_FINAL):
        u, b, n = draw_fixed_noise(mu_final, SIGMA_PROP, LAMBDA_, rng)
        fw, sig, dg = compute_fwhm_and_dgamma(
            gamma_final, u.numpy(), b.numpy(),
            _fit_fn, _fwhm_fn, _nll_fn, n_params=2)
        fwhms.append(fw); sigmas.append(sig); dfs.append(dg)
        dsigmas.append(min(2.0 / math.sqrt(max(int(n), 1)), 2.0))
        ns.append(n)

    ft = torch.tensor(fwhms, dtype=torch.float32)
    si_t = torch.tensor(sigmas, dtype=torch.float32)
    nt = torch.tensor(ns, dtype=torch.float32)
    dg_t = torch.tensor(dfs, dtype=torch.float32)
    ds_t = torch.tensor(dsigmas, dtype=torch.float32)

    # 2D joint (FWHM, σ)
    s_mu, s_gamma, _, _ = kde_scores(ft, si_t, nt, dg_t, ds_t,
                                     target_f, target_s, H_F, H_S, mu_final)
    s = torch.stack([s_mu, s_gamma], dim=1)              # (N, 2)
    J_list.append(s.T @ s / len(target_f))               # (2, 2)

    # 1D FWHM-only (degeneracy diagnostic)
    s_mu_1, s_gamma_1, _ = kde_scores_1d(ft, nt, dg_t, target_f, H_F, mu_final)
    s1 = torch.stack([s_mu_1, s_gamma_1], dim=1)
    J1_list.append(s1.T @ s1 / len(target_f))

J = torch.stack(J_list).mean(dim=0)
J_std = torch.stack(J_list).std(dim=0)
J1 = torch.stack(J1_list).mean(dim=0)

Jinv = torch.linalg.inv(J + 1e-8 * torch.eye(2))
std_mu = math.sqrt(Jinv[0, 0])
std_gamma = math.sqrt(Jinv[1, 1])
corr = Jinv[0, 1] / math.sqrt(Jinv[0, 0] * Jinv[1, 1])
eigs = torch.linalg.eigvalsh(J)
eigs_1d = torch.linalg.eigvalsh(J1)

print(f"=== FISHER INFORMATION at (μ̂, γ̂) = ({mu_final:.2f}, {gamma_final:.2f}) ===")
print(f"  J (2D joint, avg over {FISHER_SEEDS} seeds x M={M_FINAL} runs):")
print(f"    [[{J[0,0]:9.4f}, {J[0,1]:9.4f}]")
print(f"     [{J[1,0]:9.4f}, {J[1,1]:9.4f}]]")
print(f"  J spread across seeds: {J_std[0,0]:.4f} (μμ), {J_std[1,1]:.4f} (γγ)")
print(f"  CRB std:  σ_μ = {std_mu:.2f}   σ_γ = {std_gamma:.2f}   corr = {corr:+.2f}")
print(f"  Eigenvalues 2D: {eigs[0]:.4f}, {eigs[1]:.4f}  (ratio {eigs[1]/eigs[0]:.1f})")
print(f"  Eigenvalues 1D: {eigs_1d[0]:.4f}, {eigs_1d[1]:.4f}  (ratio {eigs_1d[1]/eigs_1d[0]:.1f})")
print(f"  Degeneracy diagnostic: expect a LARGE 1D eigenvalue ratio and a SMALL 2D one.")

In [ ]:
from scipy.stats import gaussian_kde
from src.losses import wasserstein_loss

steps = [h['step'] for h in history]
mu_hist = [h['mu'] for h in history]
gamma_hist = [h['gamma'] for h in history]

# Fresh forward pass at the recovered parameters for the distribution comparison
rng_f = np.random.default_rng(999)
final_fwhms = []
for _ in range(500):
    u, b, n = draw_fixed_noise(mu_hist[-1], SIGMA_PROP, LAMBDA_, rng_f)
    photons = build_photons(torch.tensor(gamma_hist[-1], dtype=torch.float32), u, b)
    theta = fit_profile(photons, n_iters=80, model='lorentzian', uniform_bg=False)
    if theta is not None:
        final_fwhms.append(fwhm_from_theta(theta, model='lorentzian').item())
    else:
        final_fwhms.append(2.0 * gamma_hist[-1])
final_t = torch.tensor(final_fwhms, dtype=torch.float32)
w1_final = wasserstein_loss(final_t, target_f).item()

def cov_to_ellipse(cov2, level=1.0):
    """Points of the Gaussian ellipse from a 2x2 covariance matrix."""
    eigvals, eigvecs = np.linalg.eigh(cov2)
    order = eigvals.argsort()[::-1]
    eigvals, eigvecs = eigvals[order], eigvecs[:, order]
    theta = np.linspace(0, 2 * np.pi, 100)
    xy = np.vstack([level * np.sqrt(eigvals[0]) * np.cos(theta),
                    level * np.sqrt(eigvals[1]) * np.sin(theta)])
    return eigvecs @ xy

Jinv_np = Jinv.numpy()

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

ax = axes[0, 0]
ax.axhline(NBAR_TRUE, color='g', ls='--', linewidth=1.5, alpha=0.7, label=f'True μ={NBAR_TRUE}')
ax.plot(steps, mu_hist, 'b-', linewidth=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('μ')
ax.set_title('μ Convergence (weighted REINFORCE)'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 1]
ax.axhline(GAMMA_TRUE, color='g', ls='--', linewidth=1.5, alpha=0.7, label=f'True γ={GAMMA_TRUE}')
ax.plot(steps, gamma_hist, 'r-', linewidth=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('γ')
ax.set_title('γ Convergence (KDE chain + implicit)'); ax.legend(); ax.grid(alpha=0.3)

ax = axes[0, 2]
ax.plot(steps, [h['nll'] for h in history], 'k-', linewidth=2)
ax.set_xlabel('Iteration'); ax.set_ylabel('NLL')
ax.set_title('KDE Negative Log-Likelihood'); ax.grid(alpha=0.3)

ax = axes[1, 0]
ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.plot(steps, [h['grad_mu'] for h in history], 'b-', linewidth=1.5)
ax.set_xlabel('Iteration'); ax.set_ylabel('∇μ')
ax.set_title('μ Gradient'); ax.grid(alpha=0.3)

ax = axes[1, 1]
ax.axhline(0, color='gray', ls='--', alpha=0.5)
ax.plot(steps, [h['grad_gamma'] for h in history], 'r-', linewidth=1.5)
ax.set_xlabel('Iteration'); ax.set_ylabel('∇γ')
ax.set_title('γ Gradient'); ax.grid(alpha=0.3)

ax = axes[1, 2]
ax.plot(mu_hist, gamma_hist, 'b.-', linewidth=1.5, markersize=8)
ax.plot(mu_hist[0], gamma_hist[0], 'go', markersize=10, label=f'Start ({MU_INIT},{GAMMA_INIT})')
ax.plot(mu_hist[-1], gamma_hist[-1], 'ro', markersize=10, label=f'End ({mu_hist[-1]:.1f},{gamma_hist[-1]:.1f})')
ax.plot(NBAR_TRUE, GAMMA_TRUE, 'k*', markersize=15, label=f'True ({NBAR_TRUE},{GAMMA_TRUE})')
xy1 = cov_to_ellipse(Jinv_np, level=1.0)
xy2 = cov_to_ellipse(Jinv_np, level=2.0)
ax.plot(xy1[0] + mu_hist[-1], xy1[1] + gamma_hist[-1], 'r--', alpha=0.7, linewidth=1.5)
ax.plot(xy2[0] + mu_hist[-1], xy2[1] + gamma_hist[-1], 'r:', alpha=0.5, linewidth=1.5)
ax.set_xlabel('μ (mean photon count)'); ax.set_ylabel('γ (HWHM MHz)')
ax.set_title('Trajectory + CRB ellipse (1σ, 2σ)'); ax.legend(); ax.grid(alpha=0.3)

plt.suptitle('12d: Joint Optimization with 2D KDE Likelihood + Fisher Information', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('fig_12d_optimization_path.png', dpi=150, bbox_inches='tight')
plt.show()

x_grid = np.linspace(0, 100, 500)
kde_target = gaussian_kde(target_f.numpy())
kde_final = gaussian_kde(final_t.numpy())
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(x_grid, kde_target(x_grid), 'k-', linewidth=2.5, label=f'Target (μ={NBAR_TRUE}, γ={GAMMA_TRUE})')
ax.plot(x_grid, kde_final(x_grid), 'r-', linewidth=2.5, label=f'Final (μ={mu_hist[-1]:.1f}, γ={gamma_hist[-1]:.1f}, W1={w1_final:.1f})')
ax.fill_between(x_grid, kde_final(x_grid), kde_target(x_grid), alpha=0.12, color='gray')
ax.set_xlabel('FWHM (MHz)'); ax.set_ylabel('Density')
ax.set_title('FWHM Distribution: Optimization Result')
ax.legend(fontsize=11); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('fig_12d_fwhm_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Target FWHM: {target_f.mean():.1f} ± {target_f.std():.1f}')
print(f'Final  FWHM: {final_t.mean():.1f} ± {final_t.std():.1f}')

## Conclusion & Next Steps

- 12d replaces 12c's W1/mean losses with a proper 2D KDE **negative log-likelihood** (MLE), so the Fisher information / Cramér–Rao bound is a valid uncertainty statement.
- The Fisher matrix at (μ̂, γ̂) gives CRB standard errors and a confidence ellipse; the 1D-only version exposes the μ/γ degeneracy that the σ dimension resolves.
- Next: compare the CRB against bootstrap spreads, run a bandwidth-sensitivity study, then apply the same likelihood machinery to the real experimental data.